In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
import gradio as gr
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# -----------------------------
# CONFIG
# -----------------------------
CHROMA_PATH = "chroma_db"
COLLECTION_NAME = "hr_collection"

# -----------------------------
# EMBEDDINGS + LLM
# -----------------------------
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# -----------------------------
# CONNECT TO CHROMA DB
# -----------------------------
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings_model,
    persist_directory=CHROMA_PATH,
)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# -----------------------------
# CHATBOT RESPONSE FUNCTION
# -----------------------------
def stream_response(message, history):

    # Retrieve relevant chunks
    docs = retriever.invoke(message)

    knowledge = ""
    for doc in docs:
        knowledge += doc.page_content + "\n\n"

    # Build RAG prompt
    rag_prompt = f"""
You are an HR assistant. 
Answer ONLY using the information in the "Knowledge" section.
Do NOT use your own internal knowledge.
Do NOT mention the knowledge section to the user.

Question: {message}

Conversation history: {history}

Knowledge:
{knowledge}
"""

    partial_message = ""

    # Stream response
    for response in llm.stream(rag_prompt):
        partial_message += response.content
        yield partial_message


# -----------------------------
# GRADIO CHATBOT UI
# -----------------------------
chatbot = gr.ChatInterface(
    stream_response,
    textbox=gr.Textbox(
        placeholder="Ask your HR question...",
        container=False,
        autoscroll=True,
        scale=7
    ),
)

chatbot.launch()


c:\AI Projects\RAG_HR_Assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Lavanya Rajesh\AppData\Local\Temp\ipykernel_17604\1664847512.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma
C:\Users\Lavanya Rajesh\AppData\Local\Temp\ipykernel_17604\1664847512.py:23: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chro

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
